# LeetCode #1076: Project Employees II

https://leetcode.com/problems/project-employees-ii/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (nested subquery)** | $O(n^2)$ | $O(n)$ |
| **Optimal: Aggregation + HAVING/Rank ★** | $O(n \log n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force (nested subquery)
Count employees per project in an inner query, then in an outer query compare each project's count against the maximum using a correlated subquery. Each outer row triggers a full inner scan — $O(n^2)$.

### Optimal: Aggregation + HAVING/Rank ★
GROUP BY project_id to count employees per project, then use a CTE or subquery to find the maximum count, and filter to only projects with that count. A single GROUP BY pass computes all counts at once; a second pass finds the maximum and selects matching projects. Total work is $O(n \log n)$ due to grouping.

**Constraints:**
* `Project` table: project_id, employee_id (composite PK)
* `Employee` table: employee_id (PK), name, experience_years
* Multiple projects can tie for the most employees

## Solutions
### C#

In [ ]:
// SQL only — aggregate employee counts, then return projects tied at the maximum
public class Solution {
    public string GetQuery() => @"
WITH ProjectCounts AS (
    SELECT project_id, COUNT(employee_id) AS emp_count
    FROM Project
    GROUP BY project_id
)
SELECT project_id
FROM ProjectCounts
WHERE emp_count = (SELECT MAX(emp_count) FROM ProjectCounts);
";
}

### Python

In [ ]:
# Pandas equivalent: count employees per project, return those tied at the max
import pandas as pd

def project_employees_ii(project: pd.DataFrame, employee: pd.DataFrame) -> pd.DataFrame:
    # Count employees per project (no join needed — project table is the link)
    counts = project.groupby('project_id')['employee_id'].count().reset_index()
    counts.columns = ['project_id', 'emp_count']
    # Return only projects whose count equals the global maximum
    max_count = counts['emp_count'].max()
    result = counts[counts['emp_count'] == max_count][['project_id']]
    return result

### Go

In [ ]:
// Pseudocode: aggregate employee count per project, return projects at the max
package main

import "fmt"

func projectEmployeesII(projects [][]int) []int {
    // Count employees per project
    counts := make(map[int]int)
    for _, row := range projects {
        projectID := row[0]
        counts[projectID]++
    }
    // Find the maximum employee count
    maxCount := 0
    for _, c := range counts {
        if c > maxCount {
            maxCount = c
        }
    }
    // Collect all projects tied at the maximum
    var result []int
    for pid, c := range counts {
        if c == maxCount {
            result = append(result, pid)
        }
    }
    fmt.Println(result)
    return result
}

### Rust

In [ ]:
// Pseudocode: aggregate employee count per project, return projects at the max
use std::collections::HashMap;

fn project_employees_ii(projects: Vec<(i32, i32)>) -> Vec<i32> {
    // Count employees per project
    let mut counts: HashMap<i32, i32> = HashMap::new();
    for (project_id, _employee_id) in &projects {
        *counts.entry(*project_id).or_insert(0) += 1;
    }
    // Find the maximum employee count across all projects
    let max_count = counts.values().copied().max().unwrap_or(0);
    // Return projects whose count matches the maximum
    let mut result: Vec<i32> = counts
        .iter()
        .filter(|(_, &c)| c == max_count)
        .map(|(&pid, _)| pid)
        .collect();
    result.sort();
    result
}

## Example Scenarios

**1. Common Case** — One project leads

**Input:** Project: [(1, E1),(1, E2),(1, E3),(2, E1),(2, E4)]; project 1 has 3, project 2 has 2
The GROUP BY count gives project 1 → 3, project 2 → 2. MAX = 3, so only project 1 is returned: `[1]`.

**2. Slightly Complex** — Two projects tie

**Input:** Project: [(1, E1),(1, E2),(2, E3),(2, E4)]
Both project 1 and project 2 have 2 employees. MAX = 2, both match. Result: `[1, 2]` (order may vary).

**3. Edge Case: Time Factor** — Large number of projects, single winner

**Input:** 1,000 projects each with varying employee counts; one project has 500 employees, all others have fewer
The GROUP BY scans all rows once. The MAX subquery scans the 1,000 group results once. Total is $O(n \log n)$ for the sort implicit in GROUP BY.

**4. Edge Case: Space Factor** — All projects tied

**Input:** 100 projects each with exactly 5 employees
Every project is returned. The intermediate `ProjectCounts` CTE holds 100 rows; the result set holds 100 rows. Space is $O(n)$ where $n$ = number of projects.

**5. Almost-Impossible but Plausible** — Single project

**Input:** Only one project (project_id = 1) with 1 employee
MAX = 1, and only one project matches it. Result: `[1]`. Both the GROUP BY and the MAX subquery degenerate to single-row operations.